In [1]:
import pymongo
from pymongo import MongoClient
import pandas as pd
import numpy as np
import re

In [ ]:
CONNECTION_STRING = "mongodb+srv://<username>:<password>@cluster0.tehe2.azure.mongodb.net/?retryWrites=true&w=majority"

In [3]:
client = MongoClient(CONNECTION_STRING)

In [4]:
db = client['Data_Engineer']

In [5]:
collection = db['Comments']

In [6]:
cursor = collection.find()

In [7]:
list_cur = list(cursor)

In [8]:
list_cur[:1]

[{'_id': ObjectId('555466761a77ce5d65ea062a'),
  'ranking': 0,
  'layer': 0,
  'user_id': 12,
  'like': 1,
  'user_email': 'hoaianh2210@gmail.com',
  'comment_on': 'event',
  'ip': 'null',
  'user_fullname': 'Nguyễn Hoài Anh',
  'publish_status': 1,
  'object_id': '552b2f0817dc135c89d7d6a2',
  'content': 'Chương trình \\ Tuyệt cú mèo\\ luôn, chờ lâu rồi mới có số mới, hóng hóng',
  'reviewer': 'gianpt',
  'report_count': 0,
  'review_status': 1,
  'timestamp': 1431595030,
  'device': 'web-playfpt',
  'report_reason': [],
  'dislike': 0,
  'comment_status': 0,
  'device_id': 'null'}]

In [9]:
df = pd.DataFrame(list_cur)

In [10]:
df.head(5)

,_id,ranking,layer,user_id,like,user_email,comment_on,ip,user_fullname,publish_status,...,content,reviewer,report_count,review_status,timestamp,device,report_reason,dislike,comment_status,device_id
0,555466761a77ce5d65ea062a,0,0,12,1,hoaianh2210@gmail.com,event,null,Nguyễn Hoài Anh,1,...,"Chương trình \ Tuyệt cú mèo\ luôn, chờ lâu rồi...",gianpt,0,1,1431595030,web-playfpt,[],0,0,null
1,555457bf1a77ce5d65ea0617,0,0,130,5,gian.phan@gmail.com,event,null,Phan Thanh Gian,1,...,Chờ đợi chương trình này đã lâu! Hẹn mọi người...,gianpt,0,1,1431591263,web-playfpt,[],0,0,null
2,555413001a77ce5d65ea0609,0,0,527201,0,cloudytang@gmail.com,vod,42.116.8.164,Đàm Ngọc Vân,0,...,comment cai,vuluc88,0,1,1431573662,android,[],0,2,983b5151a74de256
3,555357f71a77ce5d65ea0606,0,0,424467,1,n_x_cuong@yahoo.com,vod,null,nguyen xuan cuong,1,...,"lồng tiếng mà như tập đọc, nghe không nuốt nổi...",vuluc88,0,1,1431525776,web-playfpt,[],0,0,null
4,555491181a77ce5d65ea0636,0,0,805,12,lequy1579@gmail.com,vod,42.117.66.148,Quang Quỳnh Lê,1,...,Phim rất hay ;),vuluc88,0,1,1431605947,android,[],0,0,2e190e1227743c0d


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 885169 entries, 0 to 885168
Data columns (total 21 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   _id             885169 non-null  object
 1   ranking         885169 non-null  int64 
 2   layer           885169 non-null  int64 
 3   user_id         885169 non-null  int64 
 4   like            885169 non-null  int64 
 5   user_email      885169 non-null  object
 6   comment_on      885169 non-null  object
 7   ip              885169 non-null  object
 8   user_fullname   885169 non-null  object
 9   publish_status  885169 non-null  int64 
 10  object_id       885169 non-null  object
 11  content         885169 non-null  object
 12  reviewer        885169 non-null  object
 13  report_count    885169 non-null  int64 
 14  review_status   885169 non-null  int64 
 15  timestamp       885169 non-null  int64 
 16  device          885169 non-null  object
 17  report_reason   885169 non-nu

In [12]:
#Remove unused columns
df = df.drop([ 
'ranking','layer','like','user_email','user_fullname','reviewer','report_count',
'review_status','report_reason','dislike','comment_status','device_id', 
'_id'], axis=1)

In [13]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [14]:
df.head(5)

,user_id,comment_on,ip,publish_status,object_id,content,timestamp,device
0,12,event,null,1,552b2f0817dc135c89d7d6a2,"Chương trình \ Tuyệt cú mèo\ luôn, chờ lâu rồi...",1970-01-01 00:00:01.431595030,web-playfpt
1,130,event,null,1,552b2f0817dc135c89d7d6a2,Chờ đợi chương trình này đã lâu! Hẹn mọi người...,1970-01-01 00:00:01.431591263,web-playfpt
2,527201,vod,42.116.8.164,0,5453639f17dc1371f538a4cd,comment cai,1970-01-01 00:00:01.431573662,android
3,424467,vod,null,1,551a43b417dc135c58f3c691,"lồng tiếng mà như tập đọc, nghe không nuốt nổi...",1970-01-01 00:00:01.431525776,web-playfpt
4,805,vod,42.117.66.148,1,5549cace17dc132a0cee1801,Phim rất hay ;),1970-01-01 00:00:01.431605947,android


Pre-trained PhoBERT Model

In [15]:
pip install transformers torch

In [16]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

d:\Anaconda\envs\py\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
#Load pre-trained model
model = AutoModelForSequenceClassification.from_pretrained("wonrax/phobert-base-vietnamese-sentiment")

#Load Tokenizer from model 
tokenizer = AutoTokenizer.from_pretrained("wonrax/phobert-base-vietnamese-sentiment")

In [18]:
#Test
sentence = 'coi phim buồn mấy cũng phải cười vỡ bụng'

input_ids = torch.tensor([tokenizer.encode(sentence, add_special_tokens=True)])

with torch.no_grad():
    out = model(input_ids)
    print(pd.Series(out.logits.softmax(dim=-1).tolist()[0], index = ['NEG', 'POS', 'NEU']))

NEG    0.984004
POS    0.005342
NEU    0.010653
dtype: float64


In [ ]:
#Create a function to use the model: 0-negative, 1-positive, and 2-neutral or undetermined.
def sentiment(sentence):
    try:
        input_ids = tokenizer.encode(sentence, truncation=True, max_length= tokenizer.model_max_length)
        print(f"Encoded sentence: {input_ids}")  
    except Exception as e:
        print(f"Error encoding sentence: {e}")
        return 2 
    
    input_ids = torch.tensor([input_ids])
    if input_ids.shape[1] > tokenizer.model_max_length:
        print("Input sentence exceeds maximum length.")
        return 2
    with torch.no_grad():
        out = model(input_ids)
        sentiment_score = np.argmax(out.logits.softmax(dim=-1).tolist()[0])
        print(f"Predicted sentiment: {sentiment_score}")  
        return sentiment_score

In [ ]:
df_small = df.copy()
df_small.loc[:, 'sentiment_score'] = df_small['content'].apply(sentiment)

In [ ]:
df_small.to_csv('output.csv', index=False)